# Hướng Dẫn Giải Thích Chi Tiết: `src/data_loader.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của tệp `src/data_loader.py`. File này đóng vai trò cực kỳ quan trọng trong khâu chuẩn bị dữ liệu đầu vào cho toàn bộ hệ thống dự báo.

---

## 🔍 1. Tổng Quan Về Các Thư Viện Sử Dụng

Trong `data_loader.py`, chúng ta nhập các thư viện chính:
- `yfinance as yf`: Tải dữ liệu chứng khoán trực tuyến từ Yahoo Finance.
- `pandas as pd` và `numpy as np`: Xử lý và tính toán bảng dữ liệu.
- `pandas_ta as ta`: Thư viện tính toán các chỉ báo phân tích kỹ thuật.
- `urllib.request`: Gửi yêu cầu HTTP đến DNSE API để tải dữ liệu lịch sử của VNM.

In [ ]:
import sys
import os
# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))

from src.data_loader import fetch_and_prepare_data, format_vn
print("Import thành công!")

## ⚙️ 2. Công Dụng Của Các Hàm Trong `data_loader.py`

### Hàm `format_vn(value)`
- **Mục đích:** Định dạng một số thực thành chuỗi hiển thị tiền tệ VNĐ (ngăn cách hàng nghìn bằng dấu chấm và thêm đuôi VNĐ).
- **Ví dụ:** `format_vn(65000.5)` -> `"65.000,50 VNĐ"`.

In [ ]:
price = 15427452.12
print(f"Giá gốc: {price}")
print(f"Sau định dạng: {format_vn(price)}")

#### Bước B: Tải bổ sung dữ liệu lịch sử trước 2019 của Vinamilk
Dữ liệu trường chỉ bắt đầu từ năm 2019. Để huấn luyện Transformer sâu hơn, hàm sử dụng DNSE Chart API để tải thêm dữ liệu từ năm 2012. API sử dụng Epoch Timestamp để yêu cầu dữ liệu:
`https://services.entrade.com.vn/chart-api/v2/ohlcs/stock?from={start_epoch}&to={end_epoch}&symbol=VNM&resolution=1D`

**ĐỒNG BỘ MÚI GIỜ CỤC BỘ (Asia/Ho_Chi_Minh):** DNSE Chart API trả về thời gian dạng Unix millisecond epoch. Hệ thống sử dụng thư viện `pytz` để chuyển đổi Unix Epoch trực tiếp sang múi giờ `Asia/Ho_Chi_Minh` trước khi chuyển thành kiểu Date của Pandas. Việc này giúp loại bỏ hoàn toàn các lỗi lệch ngày (lệch 7 tiếng so với UTC) khi hợp nhất dữ liệu từ nhiều nguồn khác nhau.

#### Bước C: Hợp nhất và loại bỏ trùng lặp
Dữ liệu được sắp xếp theo thời gian tăng dần, loại bỏ các ngày trùng lặp giữa các nguồn bằng hàm `.drop_duplicates(subset=['date'])`.


In [ ]:
# Thử tải dữ liệu mẫu cho Vinamilk và hiển thị bảng dữ liệu
df_sample = fetch_and_prepare_data("VNM.VN", start_date="2023-01-01", end_date="2023-06-01")
print(f"Kích thước dữ liệu mẫu: {df_sample.shape}")
print("\nCác cột dữ liệu và chỉ báo kỹ thuật đã tạo:")
print(df_sample.columns.tolist())

# Hiển thị 5 dòng đầu tiên
df_sample[['date', 'open', 'close', 'ema_14', 'rsi_14', 'adx_14', 'market_return']].head()

## 📈 3. Trực Quan Hóa Chỉ Báo Đã Tính Toán

Hãy vẽ biểu đồ giá đóng cửa kèm theo đường EMA_14 và chỉ báo ADX_14 vừa được sinh ra.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(14, 7))

# Biểu đồ giá và EMA
plt.subplot(2, 1, 1)
plt.plot(df_sample['date'], df_sample['close'], label='Giá Đóng Cửa (VNĐ)', color='blue')
plt.plot(df_sample['date'], df_sample['ema_14'], label='EMA 14 Phiên', color='orange', linestyle='--')
plt.title('Giá Cổ Phiếu VNM & Chỉ Báo EMA 14')
plt.legend()
plt.grid(True)

# Biểu đồ ADX (Độ mạnh xu hướng)
plt.subplot(2, 1, 2)
plt.plot(df_sample['date'], df_sample['adx_14'], label='ADX 14', color='purple')
plt.axhline(25, color='red', linestyle=':', label='Mốc Xu Hướng Mạnh (>25)')
plt.title('Chỉ Báo Độ Mạnh Xu Hướng (ADX 14)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()